# Рекомендательная система книг

Данные: Book-Crossing (`Books_1.csv`, `Ratings.csv`). Файлы должны лежать в одной папке с ноутбуком. Ноутбук состоит из трёх частей:
1. **API (задание 4.1)** — функции, которые возвращают рекомендации;
2. **БД (дополнительно)** — SQLite, куда сохраняются оценки пользователя;
3. **Приложение (задание 4.2)** — графический интерфейс, который использует только функции API.

## 0. Установка библиотек (после установки перезапустите ядро: Restart)

In [ ]:
%pip install -q pandas scikit-learn ipywidgets

## 1. Загрузка данных и база данных

В `Ratings.csv` оценка `0` означает, что пользователь книгу прочитал, но не оценил. Такие записи убираем и оставляем только оценки 1–10.

In [ ]:
import os, sqlite3
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output

books = pd.read_csv('Books_1.csv' if os.path.exists('Books_1.csv') else 'Books.csv', low_memory=False)
books = books[['ISBN', 'Book-Title', 'Book-Author']].drop_duplicates('ISBN').set_index('ISBN')
ratings = pd.read_csv('Ratings.csv')
ratings = ratings[(ratings['Book-Rating'] > 0) & ratings.ISBN.isin(books.index)]      # только явные оценки 1-10

# --- БД: сюда сохраняются оценки, поставленные в приложении ---
conn = sqlite3.connect('book_ratings.db', check_same_thread=False)
conn.execute('CREATE TABLE IF NOT EXISTS book_my_ratings(user_id INT, isbn TEXT, rating REAL, PRIMARY KEY(user_id, isbn))')

def all_ratings():
    # оценки из датасета + оценки из БД (оценки из БД главнее)
    mine = pd.read_sql('SELECT user_id, isbn, rating FROM book_my_ratings', conn)
    mine.columns = ['User-ID', 'ISBN', 'Book-Rating']
    data = pd.concat([ratings, mine]).drop_duplicates(['User-ID', 'ISBN'], keep='last')
    return data.astype({'User-ID': int, 'ISBN': str, 'Book-Rating': float})

print(len(books), 'книг,', len(ratings), 'оценок')

## 2. API (задание 4.1)

Данных очень много, поэтому для коллаборативной фильтрации берём активных пользователей (не менее 10 оценок) и книги, которые оценили не менее 20 раз. Затем строится матрица «пользователь × книга» и матрицы сходства, и работают 4 функции-рекомендации.

In [ ]:
MIN_USER_RATINGS = 10     # пользователь должен поставить не менее стольких оценок
MIN_BOOK_RATINGS = 20     # книгу должны оценить не менее стольких раз

def build():
    # Строим модель по всем оценкам (датасет + БД)
    global R, mean, C, user_sim, item_sim, weighted, content_sim
    data = all_ratings()

    # --- рейтинг для топ-10: формула IMDb  (v*R + m*c) / (v + m) ---
    stats = data.groupby('ISBN')['Book-Rating'].agg(['count', 'mean'])
    stats = stats[stats['count'] >= MIN_BOOK_RATINGS]                       # только книги с достаточным числом оценок
    m, c = stats['count'].quantile(0.90), data['Book-Rating'].mean()
    weighted = (stats['count'] * stats['mean'] + m * c) / (stats['count'] + m)

    # --- отбор активных пользователей и популярных книг ---
    uc, bc = data['User-ID'].value_counts(), data['ISBN'].value_counts()
    app_users = pd.read_sql('SELECT DISTINCT user_id FROM book_my_ratings', conn).user_id   # пользователи приложения всегда остаются
    users_ok = uc[uc >= MIN_USER_RATINGS].index.union(app_users)
    data = data[data['User-ID'].isin(users_ok) & data.ISBN.isin(bc[bc >= MIN_BOOK_RATINGS].index)]

    R = data.pivot(index='User-ID', columns='ISBN', values='Book-Rating').astype('float32')   # матрица оценок
    mean = R.mean(axis=1)                                                                     # средняя оценка пользователя
    C = R.sub(mean, axis=0).fillna(0)                                                         # оценки минус среднее, пропуски = 0
    user_sim = pd.DataFrame(cosine_similarity(C), index=C.index, columns=C.index)             # сходство пользователей
    item_sim = pd.DataFrame(cosine_similarity(C.T), index=C.columns, columns=C.columns)       # сходство книг

    # --- сходство книг по контенту: название + автор ---
    info = books.loc[R.columns]
    docs = info['Book-Title'] + ' ' + info['Book-Author'].fillna('')
    tfidf = TfidfVectorizer(stop_words='english').fit_transform(docs)
    content_sim = pd.DataFrame(cosine_similarity(tfidf), index=R.columns, columns=R.columns)

def with_titles(scores):
    # превращает Series {ISBN: балл} в таблицу с названиями
    df = scores.rename('score').round(2).to_frame().join(books[['Book-Title', 'Book-Author']])
    return df[['Book-Title', 'Book-Author', 'score']]

def top_n(scores, user_id, n=10):
    # выбираем n лучших книг, которые пользователь ещё не оценивал
    scores = scores.dropna()
    scores = scores[~scores.index.isin(R.loc[user_id].dropna().index)]
    return with_titles(scores.nlargest(n))

# ---------- 1. Топ-10 популярных ----------
def get_top10():
    return with_titles(weighted.nlargest(10))

# ---------- 2а. Коллаборативная фильтрация по пользователям ----------
def predict_user_based(user_id, k=30):
    neigh = user_sim[user_id].drop(user_id).nlargest(k).clip(lower=0)   # k самых похожих пользователей
    rated = R.loc[neigh.index].notna().astype(float)                    # кто из соседей оценил книгу
    num = neigh.dot(C.loc[neigh.index])                                 # сумма (сходство * оценка соседа)
    den = neigh.dot(rated)                                              # сумма сходств тех, кто оценил книгу
    pred = mean[user_id] + num / den.replace(0, np.nan)
    return pred.where(rated.sum() >= 2).clip(1, 10)                     # нужно минимум 2 соседа, иначе прогноз ненадёжен

def recommend_user_based(user_id):
    return top_n(predict_user_based(user_id), user_id)

# ---------- 2б. Коллаборативная фильтрация по книгам ----------
def predict_item_based(user_id):
    sim = item_sim.clip(lower=0)                                  # берём только положительное сходство
    rated = R.loc[user_id].notna().astype(float)                  # какие книги оценил пользователь
    num = sim.dot(C.loc[user_id])                                 # сходство с оценёнными * оценка
    den = sim.dot(rated)
    pred = mean[user_id] + num / den.replace(0, np.nan)
    return pred.where((sim > 0).astype(float).dot(rated) >= 2).clip(1, 10)   # нужно минимум 2 похожие оценённые книги

def recommend_item_based(user_id):
    return top_n(predict_item_based(user_id), user_id)

# ---------- 3. По контенту (название и автор) ----------
def recommend_content(isbn):
    sim = content_sim[isbn].drop(isbn)                            # похожесть на выбранную книгу
    return with_titles(sim.nlargest(10))

# ---------- 4. Персональные рекомендации ----------
def recommend_personal(user_id):
    if user_id not in R.index:                                    # новый пользователь без оценок
        return get_top10()
    both = pd.concat([predict_user_based(user_id), predict_item_based(user_id)], axis=1).mean(axis=1)
    return top_n(both, user_id)                                   # среднее двух методов

# ---------- Оценка книги (запись в БД) ----------
def save_rating(user_id, isbn, rating):
    conn.execute('INSERT OR REPLACE INTO book_my_ratings VALUES(?,?,?)', (user_id, isbn, rating))
    conn.commit()
    build()                                                       # пересчитываем модель с новой оценкой

def find_books(text):
    # поиск среди книг модели по части названия или автора
    b = books.loc[R.columns]
    mask = b['Book-Title'].str.contains(text, case=False, regex=False) | \
           b['Book-Author'].fillna('').str.contains(text, case=False, regex=False)
    return b[mask].head(30)

# ---------- Справка ----------
HELP = """
Команды и параметры:
  get_top10()                       - топ-10 популярных книг. Параметров нет.
  recommend_user_based(user_id)     - коллаборативная фильтрация по пользователям. user_id - номер пользователя.
  recommend_item_based(user_id)     - коллаборативная фильтрация по книгам. user_id - номер пользователя.
  recommend_content(isbn)           - 10 книг, похожих по названию и автору. isbn - код книги.
  recommend_personal(user_id)       - персональные рекомендации (среднее двух методов CF). user_id - номер пользователя.
  save_rating(user_id, isbn, rating)- сохранить оценку в БД. rating от 1 до 10.
  find_books(text)                  - поиск книги по части названия или автора.
"""

build()
print('В модели:', R.shape[0], 'пользователей,', R.shape[1], 'книг')
get_top10()

### Проверка API

In [ ]:
uid = int(R.notna().sum(axis=1).idxmax())             # пользователь с наибольшим числом оценок
print('По пользователям (user', uid, '):'); display(recommend_user_based(uid))
print('По книгам (user', uid, '):');        display(recommend_item_based(uid))
first = get_top10().index[0]                          # ISBN самой популярной книги
print('По контенту (похожие на', books.at[first, 'Book-Title'], '):'); display(recommend_content(first))
print('Персональные (user', uid, '):');     display(recommend_personal(uid))

## 3. Приложение с графическим интерфейсом (задание 4.2)

1. Выберите команду и нажмите **«Показать»**.
2. Чтобы оценить книгу: найдите её по названию или автору, выберите, поставьте оценку 1–10 и нажмите **«Оценить»** (сохраняется в БД).
3. Кнопка **«Справка»** показывает команды и параметры.

In [ ]:
cmd = widgets.Dropdown(description='Команда:', layout=widgets.Layout(width='550px'), options=[
    ('1. Топ-10 популярных', 'top'),
    ('2а. Коллаборативная фильтрация: по пользователям', 'user'),
    ('2б. Коллаборативная фильтрация: по книгам', 'item'),
    ('3. По контенту (нужна выбранная книга)', 'content'),
    ('4. Персональные рекомендации', 'personal')])
user = widgets.BoundedIntText(value=uid, min=1, max=10**9, description='ID пользователя:', style={'description_width': 'initial'})
search = widgets.Text(description='Поиск книги:', placeholder='название или автор')
find_btn = widgets.Button(description='Найти')
book = widgets.Dropdown(description='Книга:', layout=widgets.Layout(width='700px'))
show_btn = widgets.Button(description='Показать', button_style='primary')
rate = widgets.IntSlider(value=8, min=1, max=10, description='Оценка:')
rate_btn = widgets.Button(description='Оценить', button_style='success')
help_btn = widgets.Button(description='Справка')
out = widgets.Output()

def on_find(b):
    book.options = [(f"{r['Book-Title']} - {r['Book-Author']}", isbn) for isbn, r in find_books(search.value).iterrows()]

def on_show(b):
    with out:
        clear_output()
        c, u = cmd.value, user.value
        if c == 'top':
            display(get_top10())
        elif c == 'content':
            if book.value is None: print('Сначала найдите и выберите книгу.')
            else: display(recommend_content(book.value))
        elif u not in R.index:
            print('Пользователь', u, 'не найден в модели (нужно не менее', MIN_USER_RATINGS, 'оценок). Оцените несколько книг.')
            display(get_top10())
        else:
            display({'user': recommend_user_based, 'item': recommend_item_based, 'personal': recommend_personal}[c](u))

def on_rate(b):
    with out:
        clear_output()
        if book.value is None: print('Сначала найдите и выберите книгу.')
        else:
            save_rating(user.value, book.value, rate.value)
            print('Сохранено в БД: пользователь', user.value, ', книга', book.label, ', оценка', rate.value)

def on_help(b):
    with out:
        clear_output(); print(HELP)

find_btn.on_click(on_find); show_btn.on_click(on_show); rate_btn.on_click(on_rate); help_btn.on_click(on_help)

display(widgets.VBox([cmd, user, show_btn,
                      widgets.HBox([search, find_btn]), book, rate,
                      widgets.HBox([rate_btn, help_btn]), out]))